# ASR-2026 — Training Notebook (Kaggle)

Run this in **Save Version → Save & Run All (Commit)** mode. It will:
1. Clone the project code from GitHub.
2. Read pre-cached 16 kHz audio from the competition dataset.
3. Resume from `last.ckpt` if a previous run's checkpoint dataset is attached.
4. Train until the Kaggle session ends (~9 h), then save `last.ckpt` + `best.ckpt` to `/kaggle/working/`.

### Notebook settings (set in the right sidebar)
- **Accelerator:** GPU T4 x2 (or P100).
- **Internet:** ON (needed for `git clone` + `pip install`).
- **Add data:**
  - the competition dataset `asr-2026-spoken-numbers-recognition-challenge`.
  - (on 2nd+ run) your checkpoints-from-last-run dataset, if any.

In [ ]:
# === Configure these ===
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
# Path to your previous-run checkpoint dataset (None on the 1st run).
# Example after you upload: '/kaggle/input/asr-2026-checkpoints/last.ckpt'
RESUME_CKPT = None

In [ ]:
!pip install -q num2words jiwer soundfile librosa resampy pydub pyyaml tqdm audiomentations

In [ ]:
!git clone {GITHUB_REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo

In [ ]:
# Point the config at Kaggle-mounted paths.
import yaml, os
from pathlib import Path

def find_data_root() -> str:
    base = Path('/kaggle/input')
    candidates = [base]
    candidates.extend(p for p in base.iterdir() if p.is_dir())
    candidates.extend(p for d in base.iterdir() if d.is_dir() for p in d.iterdir() if p.is_dir())
    for path in candidates:
        if (path / 'train.csv').exists() and (path / 'dev.csv').exists() and (path / 'test.csv').exists():
            return str(path)
    raise FileNotFoundError(f'Could not find dataset root under {base}. Found: {[str(p) for p in base.iterdir()]}')

with open('configs/conformer_ctc.yaml') as f:
    cfg = yaml.safe_load(f)
DATA_ROOT = find_data_root()
print('Using DATA_ROOT =', DATA_ROOT)
cfg['data']['data_root'] = DATA_ROOT
cfg['data']['train_csv'] = f'{DATA_ROOT}/train.csv'
cfg['data']['dev_csv']   = f'{DATA_ROOT}/dev.csv'
cfg['data']['test_csv']  = f'{DATA_ROOT}/test.csv'
cfg['data']['cache_root'] = None    # Kaggle audio is already 16 kHz for dev/test; train will resample on-the-fly
cfg['paths']['out_dir']  = '/kaggle/working/runs/conformer_ctc'
cfg['train']['num_workers'] = 2      # Kaggle has limited CPU
os.makedirs(cfg['paths']['out_dir'], exist_ok=True)

# If resuming, copy last.ckpt into out_dir so train.py picks it up automatically.
if RESUME_CKPT and os.path.exists(RESUME_CKPT):
    import shutil
    shutil.copy(RESUME_CKPT, os.path.join(cfg['paths']['out_dir'], 'last.ckpt'))
    print('Resuming from', RESUME_CKPT)

with open('configs/runtime.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print('wrote runtime.yaml')

In [ ]:
# Train. Saves /kaggle/working/runs/conformer_ctc/{last,best}.ckpt
!python -m src.train --config configs/runtime.yaml

In [ ]:
# Copy checkpoints to /kaggle/working/ root so they show up as notebook outputs.
!cp /kaggle/working/runs/conformer_ctc/last.ckpt /kaggle/working/last.ckpt
!cp /kaggle/working/runs/conformer_ctc/best.ckpt /kaggle/working/best.ckpt || true
!ls -lh /kaggle/working/*.ckpt